In [ ]:
import sisl
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In this example, you will familiarize yourself with calculations with spin-polarized quantities.

To exemplify, we will do a simple system with a Carbon chain as the electrodes, and a $\mathrm{CO}$ molecule in the junction. Additionally this tutorial will focus on calculating projection transmissions, where we are projecting the scattering matrix ($\boldsymbol\Gamma$) onto the molecular eigenstates of the $\mathrm{CO}$ molecule. With this procedure one can calculate how much transmission that goes through a particular orbital. For additional details, please have a look in [Advanced 5](../A_05/A_05.ipynb).

In `sisl` we need to estimate the atomic configuration of the molecule and chains. Here we will use the van der Walls radius for the atomic species and also their calculated radius (see [here](https://en.wikipedia.org/wiki/Atomic_radii_of_the_elements_(data_page))).

The CO molecule will be placed in the middle of the junction with some offset (to induce some defect).

In [ ]:
chain = sisl.Geometry([C_R, 0, 0], lattice=[C_R * 2, 10, 10], atoms=sisl.Atom(6))
elec = chain.tile(4, axis=0)
elec.write("STRUCT_ELEC.fdf")

In [ ]:
# To estimate the distance between two atoms, we'll use the estimation from the vdw table radius for the atomic species.
# This is because we are doing weak transport (due to the molecular projection interest).
PT = sisl.PeriodicTable()

C_R = PT.radius("C")
C_vdw_R = PT.radius("C", method="vdw")
O_R = PT.radius("O")
O_vdw_R = PT.radius("O", method="vdw")

# To estimate the distance between two atoms, we'll use the estimation from the vdw table radius for the atomic species.
# This is because we are doing weak transport (due to the molecular projection interest).
PT = sisl.PeriodicTable()

# The molecule with appropriate distance.
CO = sisl.Geometry(
    [[C_vdw_R + C_R * 0.7, 0.1, 0], [C_vdw_R + C_R * 1.7 + O_R, 0.1, 0]],
    lattice=[C_vdw_R + C_R * 1.7 + O_R * 1.7 + O_vdw_R, 1, 1],
    atoms=["C", "O"],
)

device = (
    elec.tile(3, axis=0)
    .append(CO, axis=0)
    .append(elec.tile(3, axis=0), axis=0)
    .translate([0, 5, 0])
)
device.write("STRUCT.fdf")
device.plot(axes="xy")

In [ ]:
%%bash
#%solution clear
rm -f siesta.TBT_*.nc
siesta RUN_ELEC.fdf > RUN_ELEC.out
siesta RUN.fdf > RUN.out
tbtrans RUN.fdf > TBT.out
echo all completed

## Exercises

- Check the `dq` and the forces on the atoms, do you think this system is physically sound?
- Perform all required TranSiesta calculations, first the electrodes, then the device region.
- Then do the TBtrans calculation.
  Note that in the `RUN.fdf` there is an additional block of the projection structure. Please have a look at [Advanced 5](../A_05/A_05.ipynb) for additional details on how this one works.
- Since in this case it is a polarized calculation, there are multiple transmission files to look into.  
  And each of these have their own transmission.
  One way to start of the data-analysis would be to plot the two spin-channels individually. This is a good starting point because different spin-components may have drastically different magnitudes. And you may accidentally miss some features because of the scale of the different spin-comoponents.

In [ ]:
tbts = [sisl.get_sile("siesta.TBT_UP.nc"), sisl.get_sile("siesta.TBT_DN.nc")]
projs = [sisl.get_sile("siesta.TBT_UP.Proj.nc"), sisl.get_sile("siesta.TBT_DN.Proj.nc")]
# The eigenstates used to do projections, can be extracted using
# sisl as well. They are part of the Proj.nc file, and can be accessed via:
print(projs[0].eigenstate("CO").eig)
spins = ["up", "down"]

In [ ]:
_, axs = plt.subplots(1, 2, figsize=(12, 8))
for i, (spin, tbt) in enumerate(zip(spins, tbts)):
    # %solution begin
    axs[i].plot(tbt.E, tbt.DOS(), label=spin)
    axs[i].plot(tbt.E, tbt.ADOS("Left"), label=spin)
    axs[i].legend();
# %solution end

In [ ]:
alpha = 0.7
_, axs = plt.subplots(2, 2, figsize=(12, 12))
for i, (spin, tbt, proj) in enumerate(zip(spins, tbts, projs)):
    axs[0][i].set_xlim([-0.5, 0.5])
    axs[0][i].set_title(spin)
    axs[1][i].set_xlim([-0.5, 0.5])

    axs[0][i].plot(tbt.E, tbt.transmission(0, 1), label=spin)
    # put in the plots of the projections, from a projection to the right electrode
    # There are no *right* projections.
    # %solution begin
    axs[1][i].plot(
        proj.E, proj.transmission("Left.CO.HOMO", "Right"), label="HOMO", alpha=alpha
    )
    axs[1][i].plot(
        proj.E, proj.transmission("Left.CO.LUMO1", "Right"), label="LUMO1", alpha=alpha
    )
    axs[1][i].plot(
        proj.E, proj.transmission("Left.CO.LUMO2", "Right"), label="LUMO2", alpha=alpha
    )
    # %solution end

    axs[0][i].set_ylim([0, None])
    axs[1][i].set_ylim([0, None])
plt.legend();